# Ü-G — DynamoDB als natives Ziel: schreiben & zurücklesen (LÖSUNG)

`raw.orders` × `raw.customers` angereichert, je Kunde eine Zeile in
`gfu-glue-training-orders-enriched` (nativ, ohne Connection), dann per
Scan-API zurückgelesen. PAY_PER_REQUEST → quasi kostenlos.

In [ ]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
# In Glue Studio wird die IAM-Rolle in der UI gewählt. Alternativ per Magic:
# %iam_role arn:aws:iam::<account>:role/AWSGlueServiceRole-GfuGlueTraining
%%configure
{ "--enable-glue-datacatalog": "true" }

In [ ]:
from awsglue.context import GlueContext
from awsglue.transforms import ApplyMapping
from pyspark.context import SparkContext
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

DDB_TABLE = "gfu-glue-training-orders-enriched"   # hash_key = customer_id

## 1) Quellen lesen & je Kunde eine Zeile bilden
`raw.orders` (Katalog aus Ü3.1) × `raw.customers`; letzte Bestellung je `customer_id`.

In [ ]:
orders = glueContext.create_dynamic_frame.from_catalog(
    database="raw", table_name="orders", transformation_ctx="orders"
).toDF()
customers = glueContext.create_dynamic_frame.from_catalog(
    database="raw", table_name="customers", transformation_ctx="customers"
).toDF()

# Spalten der Rohtabelle haben Leerzeichen -> sauber benennen/casten.
o = (orders.selectExpr("`cust id` AS customer_id",
                       "CAST(`order total` AS double) AS order_total",
                       "`order date` AS order_date"))
# je Kunde letzte Bestellung (Hash-Key muss eindeutig sein)
w = Window.partitionBy("customer_id").orderBy(col("order_date").desc())
latest = (o.withColumn("rn", row_number().over(w))
           .filter(col("rn") == 1).drop("rn"))
enriched = (latest.join(customers.select(col("customer_id"),
                                         col("name"),
                                         col("address.city").alias("city")),
                        "customer_id", "left"))
enriched.show(truncate=False)

## 2) Nach DynamoDB schreiben (nativ)
`write_dynamic_frame.from_options` mit `connection_type="dynamodb"`.
`dynamodb.throughput.write.percent` drosselt bei provisionierten Tabellen — bei
PAY_PER_REQUEST unkritisch.

In [ ]:
from awsglue.dynamicframe import DynamicFrame

# alle DynamoDB-Attributwerte als String halten ist am robustesten fürs Demo
enriched_str = enriched.selectExpr(
    "CAST(customer_id AS string) customer_id",
    "CAST(order_total AS string) order_total",
    "CAST(order_date AS string) order_date",
    "CAST(name AS string) name", "CAST(city AS string) city")
dyf = DynamicFrame.fromDF(enriched_str, glueContext, "dyf")

glueContext.write_dynamic_frame.from_options(
    frame=dyf,
    connection_type="dynamodb",
    connection_options={
        "dynamodb.output.tableName": DDB_TABLE,
        "dynamodb.throughput.write.percent": "1.0",
    },
)
print("geschrieben nach", DDB_TABLE)

## 3) Aus DynamoDB zurücklesen (Scan-API)
`create_dynamic_frame.from_options` mit `dynamodb.input.tableName`.

In [ ]:
back = glueContext.create_dynamic_frame.from_options(
    connection_type="dynamodb",
    connection_options={
        "dynamodb.input.tableName": DDB_TABLE,
        "dynamodb.throughput.read.percent": "1.0",
    },
    transformation_ctx="back",
)
back.printSchema()
back.toDF().show(truncate=False)

## Stolpersteine (Block 6)
- **Hash-Key eindeutig:** doppelte `customer_id` überschreiben sich still.
- **Scan liest die ganze Tabelle** — bei großen Tabellen teuer/langsam; hier ok.
- **Kein Schema in DynamoDB:** der Reader leitet Attribute pro Item ab; fehlende
  Attribute werden NULL. Zahlen ggf. als String schreiben, um Typ-Überraschungen
  zu vermeiden.
- **Kosten:** PAY_PER_REQUEST + Trainingsdatenmenge ≈ 0; keine stehende Kapazität.